# Milestone 2 · Multi-Step Funnel & Friction Analysis

**Project:** A/B Testing & Product Experimentation Analysis  
**Author:** Akshobya698  
**Objective:** Ingest user clickstream logs, deduplicate multi-session events, compute step-over-step conversions, and isolate mobile drop-off friction across the 5-stage purchase funnel.

**Funnel stages:** Homepage → Product Page → Cart → Checkout → Payment

## 0 · Imports & Configuration

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 20)

SEED = 42
rng  = np.random.default_rng(SEED)

STEPS   = ['Homepage', 'Product Page', 'Cart', 'Checkout', 'Payment']
DEVICES = ['Desktop', 'Mobile']

os.makedirs('../findings', exist_ok=True)
print('Setup complete.')

Setup complete.


## 1 · Synthetic Clickstream Data Generation

We simulate a raw clickstream table that mirrors production reality:
- Each user can have **multiple sessions** and **repeated pings per step** (page refresh, back-navigation).
- Device is assigned at the user level (a user is consistently Desktop or Mobile).
- Funnel drop-off is baked in — not every user reaches every stage.

In [2]:
# ── Funnel reach probabilities (cumulative) ──────────────────────────────────
# Desktop users convert better at every stage
reach_prob = {
    #            HP    PP     Cart   CK     Pay
    'Desktop': [1.00, 0.680, 0.415, 0.252, 0.189],
    'Mobile':  [1.00, 0.587, 0.323, 0.168, 0.093],
}

N_USERS        = 10_000          # unique users who hit the Homepage
N_DESKTOP      = 5_500           # 55 % desktop split
N_MOBILE       = N_USERS - N_DESKTOP
MAX_SESSIONS   = 4               # a user may return across sessions
MAX_PINGS_STEP = 5               # page refreshes / back-nav per step

rows = []
event_id = 1

for device, n_users in [('Desktop', N_DESKTOP), ('Mobile', N_MOBILE)]:
    probs = reach_prob[device]
    for uid_idx in range(n_users):
        user_id    = f"{device[0]}U{uid_idx+1:05d}"   # e.g. DU00001, MU00001
        n_sessions = rng.integers(1, MAX_SESSIONS + 1)

        for step_idx, step in enumerate(STEPS):
            # Does this user reach this step?
            if rng.random() > probs[step_idx]:
                break   # once a user exits the funnel they don't skip ahead

            # Generate 1–MAX_PINGS_STEP pings for this step (multi-session noise)
            n_pings = rng.integers(1, MAX_PINGS_STEP + 1)
            for _ in range(n_pings):
                session_id = f"{user_id}_S{rng.integers(1, n_sessions+1):02d}"
                ts_offset  = rng.integers(0, 30 * 24 * 3600)   # within 30 days
                rows.append({
                    'event_id'  : event_id,
                    'user_id'   : user_id,
                    'session_id': session_id,
                    'device'    : device,
                    'step'      : step,
                    'step_order': step_idx + 1,
                    'timestamp' : pd.Timestamp('2024-01-01') + pd.Timedelta(seconds=int(ts_offset)),
                })
                event_id += 1

raw_df = pd.DataFrame(rows)
print(f"Raw clickstream: {len(raw_df):,} events  |  "
      f"{raw_df['user_id'].nunique():,} unique users  |  "
      f"{raw_df['session_id'].nunique():,} unique sessions")
raw_df.head(10)

Raw clickstream: 58,339 events  |  10,000 unique users  |  20,437 unique sessions


,event_id,user_id,session_id,device,step,step_order,timestamp
0,1,DU00001,DU00001_S01,Desktop,Homepage,1,2024-01-13 23:46:15
1,2,DU00001,DU00001_S01,Desktop,Homepage,1,2024-01-26 18:11:25
2,3,DU00001,DU00001_S01,Desktop,Homepage,1,2024-01-03 13:52:51
3,4,DU00001,DU00001_S01,Desktop,Homepage,1,2024-01-21 22:06:17
4,5,DU00001,DU00001_S01,Desktop,Product Page,2,2024-01-30 06:26:53
5,6,DU00001,DU00001_S01,Desktop,Product Page,2,2024-01-23 01:44:30
6,7,DU00001,DU00001_S01,Desktop,Product Page,2,2024-01-23 20:01:14
7,8,DU00002,DU00002_S02,Desktop,Homepage,1,2024-01-12 02:58:28
8,9,DU00003,DU00003_S03,Desktop,Homepage,1,2024-01-14 07:15:29
9,10,DU00003,DU00003_S02,Desktop,Homepage,1,2024-01-07 19:36:42


## 2 · Data Quality & Deduplication

### Why `nunique()` instead of `count()`?

Raw clickstream logs fire one event **per page render**, not per user visit.  
A single user refreshing the Cart page 4 times produces 4 rows — if we use `count()` we inflate the Cart population and understate drop-off.  
**Rule:** count each `user_id` at most once per funnel stage (`nunique()` / dedup).

In [3]:
# ── Basic data quality checks ────────────────────────────────────────────────
print("=== Null values ===")
print(raw_df.isnull().sum())

print("\n=== Duplicate (user_id, step, session_id) pings ===")
dupes = raw_df.duplicated(subset=['user_id', 'session_id', 'step']).sum()
print(f"  {dupes:,} duplicate pings in raw table")

print("\n=== Raw event counts vs unique users per step ===")
raw_counts = raw_df.groupby('step').agg(
    raw_events = ('event_id', 'count'),
    unique_users = ('user_id', 'nunique')
).reindex(STEPS)
raw_counts['inflation_factor'] = (raw_counts['raw_events'] / raw_counts['unique_users']).round(2)
print(raw_counts)

=== Null values ===
event_id      0
user_id       0
session_id    0
device        0
step          0
step_order    0
timestamp     0
dtype: int64

=== Duplicate (user_id, step, session_id) pings ===
  25,587 duplicate pings in raw table

=== Raw event counts vs unique users per step ===
              raw_events  unique_users  inflation_factor
step                                                    
Homepage           30075         10000              3.01
Product Page       19150          6388              3.00
Cart                7319          2447              2.99
Checkout            1593           519              3.07
Payment              202            74              2.73


In [4]:
# ── Deduplication: keep earliest event per (user_id, step) ───────────────────
dedup_df = (
    raw_df
    .sort_values('timestamp')
    .drop_duplicates(subset=['user_id', 'step'], keep='first')
    .reset_index(drop=True)
)

print(f"After dedup: {len(dedup_df):,} rows  "
      f"(removed {len(raw_df)-len(dedup_df):,} duplicate pings)")
dedup_df[['event_id','user_id','session_id','device','step','step_order','timestamp']].head(8)

After dedup: 19,428 rows  (removed 38,911 duplicate pings)


,event_id,user_id,session_id,device,step,step_order,timestamp
0,38787,MU00934,MU00934_S01,Mobile,Homepage,1,2024-01-01 00:00:34
1,21979,DU03564,DU03564_S03,Desktop,Cart,3,2024-01-01 00:01:11
2,8651,DU01393,DU01393_S02,Desktop,Product Page,2,2024-01-01 00:01:51
3,10124,DU01636,DU01636_S01,Desktop,Homepage,1,2024-01-01 00:02:11
4,33334,DU05440,DU05440_S02,Desktop,Cart,3,2024-01-01 00:02:28
5,31305,DU05122,DU05122_S03,Desktop,Homepage,1,2024-01-01 00:02:41
6,39841,MU01129,MU01129_S04,Mobile,Homepage,1,2024-01-01 00:03:54
7,53306,MU03572,MU03572_S02,Mobile,Homepage,1,2024-01-01 00:04:07


## 3 · Overall Funnel Metrics

For each step we calculate:
- **Users at step** — unique users after dedup
- **Cumulative CR** — % of Homepage visitors who reached this step
- **Step-over-Step CR** — % of previous step users who advanced
- **Step Drop-off %** — complement of Step-over-Step CR

In [5]:
def build_funnel(df, label='Overall'):
    """Compute funnel metrics from a deduplicated DataFrame."""
    counts = (
        df.groupby('step')['user_id']
        .nunique()
        .reindex(STEPS)
        .rename('users')
        .reset_index()
    )
    counts['segment']        = label
    homepage_n               = counts.loc[0, 'users']
    counts['cumulative_cr']  = (counts['users'] / homepage_n * 100).round(2)
    counts['step_cr']        = (counts['users'] / counts['users'].shift(1) * 100).round(2)
    counts['step_dropoff']   = (100 - counts['step_cr']).round(2)
    counts.loc[0, ['step_cr', 'step_dropoff']] = [100.00, 0.00]   # baseline
    return counts

overall_funnel = build_funnel(dedup_df, 'Overall')
print(overall_funnel.to_string(index=False))

        step  users segment  cumulative_cr  step_cr  step_dropoff
    Homepage  10000 Overall         100.00   100.00          0.00
Product Page   6388 Overall          63.88    63.88         36.12
        Cart   2447 Overall          24.47    38.31         61.69
    Checkout    519 Overall           5.19    21.21         78.79
     Payment     74 Overall           0.74    14.26         85.74


In [6]:
# ── Sanity check: user count must be monotonically non-increasing ────────────
counts_series = overall_funnel['users'].values
monotonic_ok  = all(counts_series[i] >= counts_series[i+1] for i in range(len(counts_series)-1))
print(f"Monotonically non-increasing user counts: {'✅ PASS' if monotonic_ok else '❌ FAIL'}")

Monotonically non-increasing user counts: ✅ PASS


## 4 · Device-Segmented Funnel (Desktop vs Mobile)

In [7]:
desktop_funnel = build_funnel(dedup_df[dedup_df['device'] == 'Desktop'], 'Desktop')
mobile_funnel  = build_funnel(dedup_df[dedup_df['device'] == 'Mobile'],  'Mobile')

device_funnel = pd.concat([desktop_funnel, mobile_funnel], ignore_index=True)

# ── Sanity check: Desktop + Mobile users == Overall users at each step ────────
sum_check = (
    device_funnel.groupby('step')['users'].sum()
    .reindex(STEPS)
    .reset_index()
    .rename(columns={'users': 'device_sum'})
    .merge(overall_funnel[['step','users']].rename(columns={'users':'overall'}), on='step')
)
sum_check['match'] = sum_check['device_sum'] == sum_check['overall']
print("Device sum == Overall at every step:")
print(sum_check.to_string(index=False))
print(f"\nAll match: {'✅ PASS' if sum_check['match'].all() else '❌ FAIL'}")

Device sum == Overall at every step:
        step  device_sum  overall  match
    Homepage       10000    10000   True
Product Page        6388     6388   True
        Cart        2447     2447   True
    Checkout         519      519   True
     Payment          74       74   True

All match: ✅ PASS


In [8]:
# ── Side-by-side view ────────────────────────────────────────────────────────
pivot = device_funnel.pivot(index='step', columns='segment', values=['users','step_cr','step_dropoff'])
pivot = pivot.reindex(STEPS)
pivot.columns = ['_'.join(c).strip() for c in pivot.columns]
pivot['mobile_lag_pp'] = (pivot['step_cr_Desktop'] - pivot['step_cr_Mobile']).round(2)
print("\nDevice-segmented funnel (step-over-step CR %)  —  mobile_lag_pp = Desktop CR − Mobile CR")
print(pivot.to_string())


Device-segmented funnel (step-over-step CR %)  —  mobile_lag_pp = Desktop CR − Mobile CR
              users_Desktop  users_Mobile  step_cr_Desktop  step_cr_Mobile  step_dropoff_Desktop  step_dropoff_Mobile  mobile_lag_pp
step                                                                                                                                
Homepage            5500.00       4500.00           100.00          100.00                  0.00                 0.00           0.00
Product Page        3733.00       2655.00            67.87           59.00                 32.13                41.00           8.87
Cart                1581.00        866.00            42.35           32.62                 57.65                67.38           9.73
Checkout             378.00        141.00            23.91           16.28                 76.09                83.72           7.63
Payment               66.00          8.00            17.46            5.67                 82.54                

## 5 · Visualisation

### 5a · Overall Funnel — Plotly Funnel Chart

In [9]:
fig_funnel = go.Figure(go.Funnel(
    name      = 'All Users',
    y         = overall_funnel['step'],
    x         = overall_funnel['users'],
    textinfo  = 'value+percent initial',
    textfont  = dict(size=14),
    marker    = dict(
        color = ['#4361ee','#4895ef','#4cc9f0','#f72585','#b5179e'],
        line  = dict(width=2, color='white')
    ),
    connector = dict(line=dict(color='#dee2e6', width=1))
))

fig_funnel.update_layout(
    title     = dict(text='<b>5-Stage Purchase Funnel — All Users</b><br><sup>Unique users after deduplication · 30-day window</sup>', x=0.5),
    width     = 820,
    height    = 540,
    plot_bgcolor = 'white',
    paper_bgcolor = 'white',
    font      = dict(family='Inter, Arial', size=13),
    margin    = dict(l=80, r=80, t=100, b=60)
)

fig_funnel.write_image('../findings/funnel_overall.png', scale=2)
fig_funnel.show()
print('Saved → findings/funnel_overall.png')

Saved → findings/funnel_overall.png


### 5b · Desktop vs Mobile — Grouped Bar Chart

In [10]:
PALETTE = {'Desktop': '#4361ee', 'Mobile': '#f72585'}

fig_device = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Unique Users at Each Stage</b>', '<b>Step-over-Step Conversion Rate (%)</b>'),
    horizontal_spacing=0.12
)

for seg, seg_df in device_funnel.groupby('segment'):
    seg_df = seg_df.set_index('step').reindex(STEPS).reset_index()
    # Users bar
    fig_device.add_trace(go.Bar(
        name       = seg,
        x          = seg_df['step'],
        y          = seg_df['users'],
        marker_color = PALETTE[seg],
        text       = seg_df['users'],
        textposition = 'outside',
        showlegend = True,
    ), row=1, col=1)
    # Step CR line
    fig_device.add_trace(go.Scatter(
        name       = seg,
        x          = seg_df['step'].iloc[1:],
        y          = seg_df['step_cr'].iloc[1:],
        mode       = 'lines+markers',
        line       = dict(color=PALETTE[seg], width=3),
        marker     = dict(size=9),
        showlegend = False,
    ), row=1, col=2)

fig_device.update_layout(
    title  = dict(text='<b>Funnel Breakdown — Desktop vs Mobile</b>', x=0.5),
    width  = 1100,
    height = 520,
    barmode = 'group',
    plot_bgcolor  = 'white',
    paper_bgcolor = 'white',
    font   = dict(family='Inter, Arial', size=12),
    legend = dict(orientation='h', y=1.12, x=0.38),
    margin = dict(l=60, r=60, t=110, b=60)
)
fig_device.update_yaxes(showgrid=True, gridcolor='#f0f0f0')

fig_device.write_image('../findings/funnel_device_split.png', scale=2)
fig_device.show()
print('Saved → findings/funnel_device_split.png')

Saved → findings/funnel_device_split.png


### 5c · Drop-off Heatmap by Stage × Device

In [11]:
heatmap_df = device_funnel[device_funnel['step'] != 'Homepage'].copy()
heatmap_pivot = heatmap_df.pivot(index='segment', columns='step', values='step_dropoff')
heatmap_pivot = heatmap_pivot.reindex(columns=STEPS[1:])

fig_heat = go.Figure(go.Heatmap(
    z          = heatmap_pivot.values,
    x          = heatmap_pivot.columns.tolist(),
    y          = heatmap_pivot.index.tolist(),
    colorscale = 'RdYlGn_r',
    text       = [[f'{v:.1f}%' for v in row] for row in heatmap_pivot.values],
    texttemplate = '%{text}',
    textfont   = dict(size=16, color='white'),
    showscale  = True,
    colorbar   = dict(title='Drop-off %')
))

fig_heat.update_layout(
    title  = dict(text='<b>Step Drop-off % · Desktop vs Mobile</b><br><sup>Red = highest friction · Green = lowest</sup>', x=0.5),
    width  = 860,
    height = 340,
    font   = dict(family='Inter, Arial', size=13),
    margin = dict(l=80, r=80, t=100, b=60)
)

fig_heat.write_image('../findings/funnel_dropoff_heatmap.png', scale=2)
fig_heat.show()
print('Saved → findings/funnel_dropoff_heatmap.png')

Saved → findings/funnel_dropoff_heatmap.png


## 6 · Business Question Answers

In [12]:
# Q1 — Largest absolute drop-off
worst_step = overall_funnel.iloc[1:].sort_values('step_dropoff', ascending=False).iloc[0]
print(f"Q1 · Largest absolute drop-off: {worst_step['step']}  ({worst_step['step_dropoff']:.1f}% of previous step lost)")

# Q2 — Absolute user loss at each step
overall_funnel['users_lost'] = overall_funnel['users'].shift(1) - overall_funnel['users']
overall_funnel.loc[0,'users_lost'] = 0
biggest_loss = overall_funnel.sort_values('users_lost', ascending=False).iloc[0]
print(f"Q2 · Biggest absolute user loss: {biggest_loss['step']}  (−{int(biggest_loss['users_lost']):,} users)")

# Q3 — Mobile vs Desktop CR gap at checkout (typically the sharpest)
ck_desktop = desktop_funnel.loc[desktop_funnel['step']=='Checkout','step_cr'].values[0]
ck_mobile  = mobile_funnel.loc[mobile_funnel['step']=='Checkout','step_cr'].values[0]
print(f"Q3 · Checkout step CR  —  Desktop: {ck_desktop:.1f}%  |  Mobile: {ck_mobile:.1f}%  "
      f"(gap = {ck_desktop-ck_mobile:.1f} pp)")

pay_desktop = desktop_funnel.loc[desktop_funnel['step']=='Payment','cumulative_cr'].values[0]
pay_mobile  = mobile_funnel.loc[mobile_funnel['step']=='Payment','cumulative_cr'].values[0]
print(f"     End-to-end (Homepage→Payment)  —  Desktop: {pay_desktop:.1f}%  |  Mobile: {pay_mobile:.1f}%")

Q1 · Largest absolute drop-off: Payment  (85.7% of previous step lost)
Q2 · Biggest absolute user loss: Cart  (−3,941 users)
Q3 · Checkout step CR  —  Desktop: 23.9%  |  Mobile: 16.3%  (gap = 7.6 pp)
     End-to-end (Homepage→Payment)  —  Desktop: 1.2%  |  Mobile: 0.2%


## 7 · Export Funnel Summary CSV

In [13]:
summary = pd.concat([overall_funnel, desktop_funnel, mobile_funnel], ignore_index=True)
summary.to_csv('../findings/funnel_summary.csv', index=False)
print('Saved → findings/funnel_summary.csv')
print(summary.to_string(index=False))

Saved → findings/funnel_summary.csv
        step  users segment  cumulative_cr  step_cr  step_dropoff  users_lost
    Homepage  10000 Overall         100.00   100.00          0.00        0.00
Product Page   6388 Overall          63.88    63.88         36.12     3612.00
        Cart   2447 Overall          24.47    38.31         61.69     3941.00
    Checkout    519 Overall           5.19    21.21         78.79     1928.00
     Payment     74 Overall           0.74    14.26         85.74      445.00
    Homepage   5500 Desktop         100.00   100.00          0.00         NaN
Product Page   3733 Desktop          67.87    67.87         32.13         NaN
        Cart   1581 Desktop          28.75    42.35         57.65         NaN
    Checkout    378 Desktop           6.87    23.91         76.09         NaN
     Payment     66 Desktop           1.20    17.46         82.54         NaN
    Homepage   4500  Mobile         100.00   100.00          0.00         NaN
Product Page   2655  Mobile 